# 탐지기 Colab 학습 안내 (최대 30 Epoch · 중단 후 재개)

**`detector_data/` 및 수정된 `detector/inference.py`가 main에 병합된 뒤 사용하세요.**

이 Notebook은 **실제 전체 데이터 학습**용입니다. 기본값은 최대 30 Epoch이며, 검증 지표에 따른 조기 종료가 적용됩니다. Colab 런타임 중단 시 Drive에 저장된 **마지막 완료 Epoch**의 `last.pt`에서 이어서 실행할 수 있습니다.

- 처음 학습: ⑤ 셀의 `RESUME_TRAINING=False`, 기본 `MAX_EPOCHS=30`을 유지합니다.
- 이어서 학습: **동일한 데이터·설정·실행명**으로 ①~⑩을 다시 실행합니다. ⑤ 셀에서 `RESUME_TRAINING=True` 및 기존 `EXISTING_RUN_NAME`을 지정합니다.
- 2 Epoch 테스트에서 얻은 모델을 30 Epoch의 **동일 스케줄로 재개하는 절차는 아닙니다.** 처음부터 최대 30 Epoch로 시작하여 중단 시 같은 설정으로 복구하세요.
- 데이터는 기본적으로 Drive에서 직접 읽습니다. `smoke_test.zip`은 전체 학습에 사용하지 않습니다.
- 데이터·환경·체크포인트를 임의 변경하지 마세요. 재개 기능은 코드 수준으로 구성되었으나 **Colab 실제 중단·재접속 검증은 별도**입니다.

## ① Google Drive 연결


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ② main 코드 가져오기

기존 Colab 폴더가 작업 브랜치이면 자동 전환하지 않습니다. 새 런타임에서 실행하거나 기존 작업을 보관한 뒤 별도 경로를 사용하세요. 저장소가 비공개이면 접근 가능한 인증 설정이 필요합니다.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/auranova1125/gitkk-pill-detection.git'
BRANCH = 'main'
project_root = Path('/content/gitkk-pill-detection')

def git(*args):
    return subprocess.check_output(['git', *args], text=True).strip()

if not project_root.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    REPO_URL, str(project_root)], check=True)
else:
    actual_branch = git('-C', str(project_root), 'branch', '--show-current')
    if actual_branch != BRANCH:
        raise RuntimeError(f'기존 폴더의 브랜치가 다릅니다: {actual_branch}. 경로와 브랜치를 확인하세요.')
for name in ('detector/__init__.py', 'detector_data/detector_inputs.py'):
    if not (project_root / name).is_file():
        raise FileNotFoundError(f'브랜치에서 필요한 코드를 찾을 수 없습니다: {name}')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print('branch:', git('-C', str(project_root), 'branch', '--show-current'))
print('commit:', git('-C', str(project_root), 'rev-parse', 'HEAD'))


## ③ 의존성 및 GPU 확인


In [ ]:
%pip -q install "torchmetrics[detection]==1.9.0" pycocotools

import torch
import torchvision
import torchmetrics
print('Python:', sys.version.split()[0])
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('torchmetrics:', torchmetrics.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Colab에서 런타임 유형을 GPU로 변경한 뒤 다시 실행하세요.')
print('GPU:', torch.cuda.get_device_name(0))


## ④ 데이터 경로 확인

Drive 직접 연결은 복사 준비가 없지만 이미지 읽기가 느릴 수 있습니다. 공유 폴더는 내 드라이브에 바로가기를 추가하고 실제 경로를 확인하세요.


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/gitkk_archive/datasets")
OUTPUT_ROOT = Path("/content/drive/MyDrive/gitkk_runs/detector")

# 필요한 파일·폴더가 있는지만 확인합니다.
required_paths = [
    "images/select_aihub",
    "images/select_codeit",
    "annotations/train.json",
    "annotations/validation.json",
    "split_manifest.csv",
    "class_mapping.csv",
]

for relative in required_paths:
    path = DATA_DIR / relative
    if not path.exists():
        raise FileNotFoundError(f"경로를 확인해 주세요: {path}")
    print(f"확인: {relative}")

print("\n데이터 경로:", DATA_DIR)
print("결과 저장 경로:", OUTPUT_ROOT)
print("Drive 직접 연결 준비 완료")

## ⑤ 데이터 설정 및 학습 모드 (최대 30 Epoch / 재개)

In [ ]:
import os
import json
from datetime import datetime, timezone
from importlib import import_module
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
from detector import (
    DetectorConfig, build_classifier_batch, evaluate_model,
    load_inference_model, plot_detection_comparison, predict_detector,
    run_training, save_inference_artifacts, select_device,
    validate_detection_batch, validate_detector_data_summary,
    validate_detector_label_map,
)

# 신규 학습: False / Colab 중단 후 재개: True
RESUME_TRAINING = False
EXISTING_RUN_NAME = ''  # 재개 시 Drive의 기존 실행 폴더 이름 (경로가 아닌 이름만)
MAX_EPOCHS = 30

settings_path = Path('/content/detector_data_settings.colab.json')
settings = {
    'package_dir': str(DATA_DIR),
    'image_roots': {
        'select': str(DATA_DIR / 'images/select_aihub'),
        'codeit': str(DATA_DIR / 'images/select_codeit'),
    },
    'augmentation': {'horizontal_flip': 0.0, 'vertical_flip': 0.0,
                     'brightness': 0.0, 'contrast': 0.0},
}
settings_path.write_text(json.dumps(settings, indent=2), encoding='utf-8')
os.environ['DETECTOR_DATA_SETTINGS'] = str(settings_path)

if RESUME_TRAINING:
    if not EXISTING_RUN_NAME.strip():
        raise ValueError('재개하려면 EXISTING_RUN_NAME에 기존 실행 폴더 이름을 입력하세요.')
    run_name = EXISTING_RUN_NAME.strip()
else:
    run_name = 'ssdlite320_train_30ep_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')

data_factory_path = 'detector_data.detector_inputs:build_detector_inputs'
run_dir = OUTPUT_ROOT / run_name
resume_path = run_dir / 'last.pt' if RESUME_TRAINING else None

config = DetectorConfig(
    run_name=run_name,
    output_dir=OUTPUT_ROOT,
    label_map_path=run_dir / 'label_map.json',
    data_summary_path=run_dir / 'data_summary.json',
    learning_rate=5e-3, weight_decay=5e-4,
    score_threshold=0.25, nms_threshold=0.45,
    early_stopping_patience=7, max_epochs=MAX_EPOCHS, batch_size=8,
    resume_from=resume_path, show_progress=True,
)
if RESUME_TRAINING:
    if not run_dir.is_dir():
        raise FileNotFoundError(f'기존 실행 폴더를 찾을 수 없습니다: {run_dir}')
    if not resume_path.is_file():
        raise FileNotFoundError(f'마지막 체크포인트를 찾을 수 없습니다: {resume_path}')
else:
    if run_dir.exists() and any(run_dir.iterdir()):
        raise FileExistsError(f'신규 학습 폴더가 이미 존재합니다: {run_dir}')

device = select_device()
assert device.type == 'cuda', device
print('device:', device, '| model:', config.model_name)
print('epochs:', config.max_epochs, '| batch_size:', config.batch_size)
print('mode:', 'resume' if RESUME_TRAINING else 'new', '| run_dir:', run_dir)

## ⑥ 데이터 연결


In [ ]:
if not data_factory_path:
    raise RuntimeError("data_factory_path를 데이터 구성 모듈의 함수로 설정하세요")

module_name, function_name = data_factory_path.split(":", maxsplit=1)
data_factory = getattr(import_module(module_name), function_name)
train_loader, val_loader, label_map, data_summary = data_factory(
    config=config,
    device=device,
)

## ⑦ 네 가지 반환값 확인


In [ ]:
validate_detector_label_map(label_map)
validate_detector_data_summary(data_summary, label_map)
for name, loader, count_key in [
    ('train_loader', train_loader, 'num_train_images'),
    ('val_loader', val_loader, 'num_val_images'),
]:
    assert len(loader.dataset) == data_summary[count_key]
    assert len(loader) > 0
    print(name, ':', loader)
    print(' dataset:', type(loader.dataset).__name__, '| images:', len(loader.dataset),
          '| batches:', len(loader), '| workers:', loader.num_workers,
          '| batch_size:', loader.batch_size)
print('\nlabel_map:')
print(json.dumps(label_map, ensure_ascii=False, indent=2))
print('\ndata_summary:')
print(json.dumps(data_summary, ensure_ascii=False, indent=2))


## ⑧ 실제 배치 검증


In [ ]:
checked_batches = {}
for name, loader in [('train', train_loader), ('validation', val_loader)]:
    batch = next(iter(loader))
    validate_detection_batch(batch, label_map)
    checked_batches[name] = batch
    rows = []
    for image, target, meta in zip(batch['images'], batch['targets'], batch['meta']):
        rows.append({
            'image_id': meta['image_id'], 'shape': tuple(image.shape),
            'dtype': str(image.dtype), 'min': image.min().item(),
            'max': image.max().item(), 'bbox_count': len(target['boxes']),
            'labels': target['labels'].unique().tolist(), 'image_path': meta['image_path'],
        })
    print(name, ': 실제 배치 검증 통과')
    display(pd.DataFrame(rows))
    print('첫 이미지 boxes (XYXY):', batch['targets'][0]['boxes'][:5])
preview_batch = checked_batches['validation']


## ⑨ 정답 bbox 시각화


In [ ]:
preview_batch = next(iter(val_loader))
preview_figure = plot_detection_comparison(
    preview_batch["images"][0],
    [],
    target=preview_batch["targets"][0],
    title=f"GT image_id={preview_batch['meta'][0]['image_id']}",
)
preview_figure

## ⑩ 전체 데이터 학습 (신규 / 이어서 학습)

In [ ]:
train_loader, val_loader, checked_label_map, checked_summary = data_factory(config=config, device=device)
assert checked_label_map == label_map
assert checked_summary == data_summary

if RESUME_TRAINING:
    # 기존 실험과 다른 데이터·모델 설정이 섞이지 않도록 학습 전에 검사합니다.
    # run_training() 자체의 호환성 검사와 별개로, 저장 파일 변경을 피하기 위한 사전 검사입니다.
    from detector.checkpoint import hash_label_map, load_checkpoint, validate_checkpoint_compatibility

    saved = load_checkpoint(config.resume_from, device)
    validate_checkpoint_compatibility(
        saved,
        expected_model_name=config.model_name,
        expected_model_config={
            'num_classes': label_map['num_model_classes'],
            'pretrained_backbone': True,
            'nms_threshold': config.nms_threshold,
        },
        expected_label_map_hash=hash_label_map(label_map),
        expected_split_hash=data_summary['split_hash'],
    )
    if saved['completed_epoch'] + 1 >= config.max_epochs:
        raise ValueError('설정한 MAX_EPOCHS까지 이미 완료했습니다. 완료된 학습을 연장하는 기능은 별도 검토가 필요합니다.')
    print(f"마지막 완료 Epoch: {saved['completed_epoch'] + 1} / {config.max_epochs}; 다음 Epoch부터 재개합니다.")
else:
    if run_dir.exists() and any(run_dir.iterdir()):
        raise FileExistsError(f'이미 결과가 있는 폴더입니다: {run_dir}. ⑤부터 새 실행명으로 실행하세요.')
    print(f'새로운 학습 시작: 최대 {config.max_epochs} Epoch')

training_result = run_training(
    config, train_loader, val_loader, label_map, data_summary,
    pretrained_backbone=True, device=device,
)
training_result

**Colab 연결이 끊겼다면:** 새 세션에서 ①~④를 다시 실행한 뒤, ⑤의 `RESUME_TRAINING=True`와 `EXISTING_RUN_NAME`을 지정하고 ⑥~⑩을 재실행합니다. 같은 `MAX_EPOCHS=30`을 유지해야 합니다. `last.pt`는 완료 Epoch마다 갱신되므로 중간 배치는 다시 처리합니다. 재개 학습은 아직 실제 Colab 재접속 테스트를 거치지 않았습니다.


## ⑪ 학습 지표 그래프


In [ ]:
metrics = pd.read_csv(training_result.run_dir / "metrics.csv")
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
metrics.plot(x="epoch", y="train_loss", ax=axes[0, 0], title="Training loss")
metrics.plot(
    x="epoch",
    y=["map", "map_50", "map_75", "map_75_95"],
    ax=axes[0, 1],
    title="Validation average precision",
)
metrics.plot(
    x="epoch",
    y=["mean_average_recall", "recall_before_threshold", "recall_after_threshold"],
    ax=axes[1, 0],
    title="Validation recall",
)
metrics.plot(
    x="epoch",
    y=["mean_count_error_before_threshold", "mean_count_error_after_threshold"],
    ax=axes[1, 1],
    title="Prediction count error",
)
fig.tight_layout()

## ⑫ 저장 파일 및 누적 Epoch 확인

In [ ]:
metrics = pd.read_csv(training_result.run_dir / 'metrics.csv')
completed_epochs = training_result.completed_epoch + 1
assert 1 <= completed_epochs <= config.max_epochs
assert len(metrics) == completed_epochs, (
    f'누적 기록과 완료 Epoch가 다릅니다: metrics={len(metrics)}, completed={completed_epochs}'
)
for name, expected in [('label_map.json', label_map), ('data_summary.json', data_summary)]:
    path = training_result.run_dir / name
    assert json.loads(path.read_text(encoding='utf-8')) == expected
    print('자동 생성 파일 확인:', path)
for name in ['best.pt', 'last.pt', 'metrics.csv', 'resolved_config.json']:
    path = training_result.run_dir / name
    assert path.is_file() and path.stat().st_size > 0, path
    print('저장 확인:', path.name)
print(f'학습 완료: {completed_epochs}/{config.max_epochs} Epoch, 조기 종료={training_result.stopped_early}')

## ⑬ best 모델 복원·평가·예측 저장

저장 모델을 새로 불러와 전체 Validation 평가 후 예측을 저장합니다. 학습 중 평가와 별도의 순회이므로 시간이 추가됩니다.


In [ ]:
best_model, device = load_inference_model(
    training_result.best_checkpoint_path,
    label_map,
    device=device,
)
best_metrics = evaluate_model(
    best_model,
    val_loader,
    device,
    score_threshold=config.score_threshold,
    show_progress=config.show_progress,
    progress_description="Detector final validation",
)
inference_result = predict_detector(
    best_model,
    val_loader,
    device,
    score_threshold=config.score_threshold,
    detector_name=config.model_name,
)
save_inference_artifacts(inference_result, training_result.run_dir)
best_metrics.to_dict()

## ⑭ 정답·예측 비교


In [ ]:
first_image_id = preview_batch["meta"][0]["image_id"]
first_predictions = [
    prediction
    for prediction in inference_result.predictions
    if prediction.image_id == first_image_id
]
comparison_figure = plot_detection_comparison(
    preview_batch["images"][0],
    first_predictions,
    target=preview_batch["targets"][0],
    title=f"GT vs prediction image_id={first_image_id}",
)
visualization_dir = training_result.run_dir / "visualizations"
visualization_dir.mkdir(parents=True, exist_ok=True)
comparison_figure.savefig(
    visualization_dir / f"image-{first_image_id}.png",
    bbox_inches="tight",
)
comparison_figure

## ⑮ 최종 파일 목록


In [ ]:
for path in sorted(training_result.run_dir.rglob('*')):
    if path.is_file():
        print(path.relative_to(training_result.run_dir), '| bytes:', path.stat().st_size)
print('데이터 연결 → 배치 검증 → 학습 → best 모델 평가·예측 저장 완료')